In [18]:
import pandas as pd
import plotly.express as px

caminho = '/content/drive/MyDrive/Ciência de Dados/Dados/censo2022.xlsx'
arquivo = pd.read_excel(caminho)

dados = arquivo.iloc[2:].copy()
col_regiao = dados.columns[0]

brasil = dados.iloc[0:1].copy()       # Brasil
regioes = dados.iloc[1:6].copy()      # Regiões
estados = dados.iloc[6:33].copy()     # Estados

# display(brasil)
# display(regioes)
# display(estados)

nomes_instrucao = [
    'Sem inst./Fund. incomp.',
    'Fund. comp./Médio incomp.',
    'Médio comp./Sup. incomp.',
    'Superior completo'
]
cols_instrucao = ['Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']

In [12]:
# 1) Perfil Educacional do Brasil
br = brasil[cols_instrucao].copy()
br.columns = nomes_instrucao

br = br.T.reset_index()
br.columns = ['Nível de Instrução', 'Percentual (%)']
br['Percentual (%)'] = pd.to_numeric(br['Percentual (%)'])

fig1 = px.line(
    br,
    x='Nível de Instrução',
    y='Percentual (%)',
    markers=True,
    title='Perfil Educacional do Brasil'
)
fig1.update_layout(yaxis=dict(range=[0, br['Percentual (%)'].max() + 5]))
fig1.show()

In [19]:
# 2) Frequência de "Superior Completo" nos Estados
est = estados[[col_regiao, 'Unnamed: 4']].copy()
est.columns = ['UF', 'Superior_Completo']
est['Superior_Completo'] = pd.to_numeric(est['Superior_Completo'])

media_nacional = pd.to_numeric(brasil['Unnamed: 4'].values[0])

fig2 = px.histogram(
    est,
    x='Superior_Completo',
    nbins=8,
    title='Histograma',
    labels={'Superior_Completo': 'Superior Completo (%)'}
)
fig2.update_layout(yaxis_title="Quantidade de Estados")

fig2.add_vline(
    x=media_nacional,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Média Nacional: {media_nacional:.1f}%",
    annotation_position="top right"
)
fig2.show()

In [20]:
# 3) Comparativo de 3 Níveis nas Regiões
paralelo = regioes[[col_regiao, 'Unnamed: 1', 'Unnamed: 3', 'Unnamed: 4']].copy()
paralelo.columns = ['Região', 'Sem_Instrucao', 'Medio_Completo', 'Superior_Completo']

paralelo['Sem_Instrucao'] = pd.to_numeric(paralelo['Sem_Instrucao'])
paralelo['Medio_Completo'] = pd.to_numeric(paralelo['Medio_Completo'])
paralelo['Superior_Completo'] = pd.to_numeric(paralelo['Superior_Completo'])

paralelo['Regiao_ID'] = range(len(paralelo))

fig3 = px.parallel_coordinates(
    paralelo,
    color='Regiao_ID',
    dimensions=['Sem_Instrucao', 'Medio_Completo', 'Superior_Completo'],
    title='Níveis de Instrução por Grande Região',
    color_continuous_scale=px.colors.diverging.Tealrose
)
fig3.show()

In [22]:
# 4) Perfil de um estado

linha = estados[estados[col_regiao].str.contains('Santa Catarina', na=False, case=False)]

valores = pd.to_numeric(linha[cols_instrucao].iloc[0]).tolist()

valores.append(valores[0])

categorias_radar = nomes_instrucao + [nomes_instrucao[0]]

df_radar = pd.DataFrame(dict(
    r=valores,
    theta=categorias_radar
))

fig4 = px.line_polar(
    df_radar,
    r='r',
    theta='theta',
    line_close=True,
    title='Perfil Educacional de SC'
)
fig4.update_traces(fill='toself')
fig4.show()